#### Set up catalog, schema and volumes

In [0]:
#set the catalog, schema and volumes
catalog = 'bootcamp_students'
schema = 'madgula_sirisha_capstone'
volume = 'landing'

bts_landing_path = f"/Volumes/{catalog}/{schema}/landing/bts"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{schema}.{volume}")

Check if there are 12 raw BTS files

In [0]:

print(bts_landing_path)
files = dbutils.fs.ls(bts_landing_path)
csv_files = [f for f in files if f.name.lower().endswith(".csv")]
print(f"Found {len(csv_files)} CSV files in {bts_landing_path}")
for f in sorted(csv_files, key=lambda x: x.name):
    print(f"  {f.name:40s} {f.size / 1e6:8.1f} MB")

assert len(csv_files) == 12, f"Expected 12 monthly files, found {len(csv_files)} — check the landing path."


### Test adsb lol and land

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # adsb.lol Connectivity Test + Landing Write
# MAGIC Part 1 confirms the API is reachable from this cluster (no auth needed).
# MAGIC Part 2 does a single poll-and-land write, so you can confirm the full
# MAGIC path (API -> JSON -> Delta bronze landing table) works before wiring up
# MAGIC a continuously running polling job.
 
# COMMAND ----------
 
import requests
 
BASE_URL = "https://api.adsb.lol"
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## Part 1: Connectivity test
# MAGIC Two quick checks — a point/radius query (returns whatever's currently
# MAGIC flying near a location) and a callsign lookup (returns a specific flight,
# MAGIC if it happens to be airborne right now). Swap in a flight you know is
# MAGIC scheduled today for a more reliable callsign test.
 
# COMMAND ----------
 
# Point/radius query — Atlanta, 50nm radius. Swap in your own coordinates if
# you'd rather test near a different hub.
test_response = requests.get(f"{BASE_URL}/v2/point/33.6407/-84.4277/50", timeout=10)
test_response.raise_for_status()
test_data = test_response.json()
 
num_aircraft = len(test_data.get("ac") or [])
print(f"Connectivity OK — {num_aircraft} aircraft found near Atlanta")
if num_aircraft:
    print("Sample record:", test_data["ac"][0])

In [0]:
# COMMAND ----------
 
# Callsign lookup — replace with a real flight number in ICAO callsign format
# (e.g. "UAL123" not "UA123") if you want to test a specific flight.
callsign_response = requests.get(f"{BASE_URL}/v2/callsign/DAL1708", timeout=10)
callsign_response.raise_for_status()
callsign_data = callsign_response.json()
 
print(f"Callsign lookup status: {callsign_response.status_code}")
print(f"Aircraft found: {len(callsign_data.get('ac') or [])}")
print(callsign_data)

In [0]:
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## Part 2: Poll-and-land write
# MAGIC Writes one poll's raw response into a bronze landing Delta table, with
# MAGIC an ingestion timestamp and the query parameters used — this is the same
# MAGIC shape your real polling job will write on every cycle, just run once
# MAGIC here to confirm the table and schema work.
 
# COMMAND ----------
 
dbutils.widgets.text("catalog", "bootcamp_students")
dbutils.widgets.text("schema", "madgula_sirisha_capstone")
dbutils.widgets.text("landing_table", "adsb_lol_landing")
 
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
landing_table = dbutils.widgets.get("landing_table")
 
#spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
#spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
 
full_table_name = f"{catalog}.{schema}.{landing_table}"
 
# COMMAND ----------
 
import json
from datetime import datetime, timezone
from pyspark.sql import Row
 
def poll_and_land(lat: float, lon: float, radius_nm: int, table_name: str) -> int:
    """Polls adsb.lol for aircraft near a point and appends the raw response
    to the landing table. Returns the number of aircraft records written."""
    response = requests.get(f"{BASE_URL}/v2/point/{lat}/{lon}/{radius_nm}", timeout=10)
    response.raise_for_status()
    data = response.json()
 
    ingested_at = datetime.now(timezone.utc)
    aircraft_list = data.get("ac") or []
 
    if not aircraft_list:
        print("No aircraft returned for this query — nothing to land.")
        return 0
 
    rows = [
        Row(
            ingested_at=ingested_at,
            query_lat=lat,
            query_lon=lon,
            query_radius_nm=radius_nm,
            raw_json=json.dumps(aircraft),
        )
        for aircraft in aircraft_list
    ]
 
    df = spark.createDataFrame(rows)
    df.write.mode("append").saveAsTable(table_name)
    return len(rows)
 
# COMMAND ----------
 
rows_written = poll_and_land(lat=33.6407, lon=-84.4277, radius_nm=50, table_name=full_table_name)
print(f"Wrote {rows_written} rows to {full_table_name}")
 
# COMMAND ----------
 
# MAGIC %md
# MAGIC ## Verify the write
 
# COMMAND ----------
 
display(spark.table(full_table_name).orderBy("ingested_at", ascending=False).limit(10))
print(f"Total rows in landing table so far: {spark.table(full_table_name).count()}")
 

### Check if the credentials for open sky are working --- Open sky is not working and is not an option anymore


In [0]:
# #store the open sky credetials in databricks secrets

# dbutils.widgets.text("open sky client id", "", "Enter Value:")
# dbutils.widgets.text("open sky client secret", "", "Enter Value:")

Create scope and add the secret


In [0]:


# from databricks.sdk import WorkspaceClient

# # 1. Initialize the Databricks Workspace Client
# w = WorkspaceClient()

# # 2. Define your scope, key, and secret value
# scope_name = "opensky"
# client_id = dbutils.widgets.get("open sky client id")
# client_secret = dbutils.widgets.get("open sky client secret")

# scopes = w.secrets.list_scopes()
# scope_exists = any(s.name.lower() == scope_name.lower() for s in scopes) if scopes else False

# if scope_exists:
#     print(f"Scope '{scope_name}' already exists.")
# else:
#     print(f"Scope '{scope_name}' does not exist. Creating it now...")
#     try:
#     # Create the Databricks-backed scope
#         w.secrets.create_scope(scope=scope_name)
#         print(f"Scope '{scope_name}' created successfully!")
#     except Exception as e:
#         print(f"Error creating scope: {e}")

# # 3. Put the secret string into the scope
# w.secrets.put_secret(
#     scope=scope_name,
#     key=client_id,
#     string_value=client_secret
# )

# print(f"Successfully added/updated secret '{client_id}' in scope '{scope_name}'.")


In [0]:
dbutils.secrets.get(scope=scope_name, key=client_id)

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # OpenSky API credential test
# MAGIC Verifies the OAuth2 client-credentials exchange works and that the resulting
# MAGIC token can successfully call /api/states/all. Run this once to confirm your
# MAGIC setup before building the real polling job.

# COMMAND ----------

# import requests

# # Pull these from your secret scope rather than hardcoding once you've confirmed
# # they work — for this one-time test, pasting them directly is fine.
# CLIENT_ID = client_id
# CLIENT_SECRET = client_secret

# TOKEN_URL = "https://auth.opensky-network.org/auth/realms/opensky-network/protocol/openid-connect/token"
# STATES_URL = "https://opensky-network.org/api/states/all"

# # COMMAND ----------

# # Step 1: exchange credentials for a bearer token
# token_response = requests.post(
#     TOKEN_URL,
#     data={
#         "grant_type": "client_credentials",
#         "client_id": CLIENT_ID,
#         "client_secret": CLIENT_SECRET,
#     },
#     timeout=10
# )
# token_response.raise_for_status()
# token_data = token_response.json()

# access_token = token_data["access_token"]
# print(f"Got token, expires in {token_data['expires_in']} seconds")

# # COMMAND ----------

# # Step 2: use the token to call a small bounding box (continental US)
# # so the response is a manageable size for a first test.
# states_response = requests.get(
#     STATES_URL,
#     headers={"Authorization": f"Bearer {access_token}"},
#     params={"lamin": 25, "lomin": -125, "lamax": 49, "lomax": -66},
#     timeout=10
# )
# states_response.raise_for_status()
# states_data = states_response.json()

# num_aircraft = len(states_data.get("states") or [])
# print(f"Success — received {num_aircraft} aircraft state vectors")
# print("Sample record:", states_data["states"][0] if num_aircraft else "none")